# Module 26 — Plan-and-Execute vs ReAct

**THE ONE IDEA:** decide the whole plan **up front** and you get something
**inspectable** — a list you can log, audit, cost, and show a human before anything runs.

You also get something **stale the moment step 1 surprises you.**

| | ReAct (module 07/08) | Plan-and-Execute |
|---|---|---|
| when the plan is made | one step at a time | **all of it, first** |
| adapts to a surprise | yes, immediately | **no** — needs an explicit replan |
| auditable before running | no | **yes** |
| planner model | must be strong | strong |
| executor model | same model | **can be cheap** |

That last row is the practical win: plan with Opus, execute with a mini model.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from pydantic import BaseModel
from typing import Literal
from _providers import get_client
from _tools import run_tool

client, MODEL, _ = get_client("openai")

class Step(BaseModel):
    tool: Literal["search_policy", "calculate"]
    arg: str
    why: str
class Plan(BaseModel):
    steps: list[Step]

TASK = ("What is the early repayment charge in year 2 on a 250000 loan, "
        "and what percentage of a 74000 annual income is that?")

## Phase 1 — Plan. Nothing has run yet.

This object is the artefact. A human can read it, a cost model can price it, and an
approval gate (module 14) can block it **before** any side effect happens.

In [ ]:
s = Plan.model_json_schema(); s["additionalProperties"] = False
r = client.chat.completions.create(model=MODEL, max_tokens=600,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "plan", "strict": True, "schema": s}},
    messages=[{"role": "user", "content":
        "Produce a complete step-by-step tool plan. Use #1, #2 to refer to earlier "
        "results inside later args.\n\nTOOLS: search_policy(query), calculate(expression)"
        f"\n\nTASK: {TASK}"}])
plan = Plan.model_validate_json(r.choices[0].message.content)

for i, st in enumerate(plan.steps, 1):
    print(f"  #{i} {st.tool:14} {st.arg[:36]:36} — {st.why[:40]}")
print(f"\n{len(plan.steps)} steps, priced and reviewable BEFORE anything runs.")

## Phase 2 — Execute. No planning happens here.

The executor substitutes earlier results and calls tools. It makes **no decisions**,
which is exactly why it can be a cheaper model — or no model at all.

In [ ]:
results = {}
for i, st in enumerate(plan.steps, 1):
    arg = st.arg
    for j, prev in results.items():
        arg = arg.replace(f"#{j}", str(prev))
    key = "query" if st.tool == "search_policy" else "expression"
    results[i] = run_tool(st.tool, {key: arg})
    print(f"  #{i} {st.tool}({arg[:34]!r})\n     -> {str(results[i])[:66]}")

## Where it breaks — a surprise at step 1

Re-run with a policy keyword that does not exist. ReAct would notice and reformulate.
The plan cannot: steps 2 and 3 were written assuming step 1 succeeded.

In [ ]:
bad = Plan(steps=[Step(tool="search_policy", arg="mortgage_holiday_covid", why="lookup"),
                  Step(tool="calculate", arg="250000 * #1", why="apply the rate")])
res = {}
for i, st in enumerate(bad.steps, 1):
    arg = st.arg
    for j, prev in res.items(): arg = arg.replace(f"#{j}", str(prev))
    key = "query" if st.tool == "search_policy" else "expression"
    res[i] = run_tool(st.tool, {key: arg})
    print(f"  #{i} -> {str(res[i])[:74]}")

print("""
^ step 1 returned 'No policy matched'. Step 2 then tried to multiply by that
STRING and produced a calculation error. The plan marched on regardless, because
marching on regardless is what a plan does.

LESSON - the trade is adaptivity for inspectability.

  ReAct              decides the next action AFTER seeing the last result, so a
                     surprise is just information. It cannot be reviewed up
                     front, and it can wander (module 12, FM3).

  Plan-and-Execute   the plan is an OBJECT. Log it, cost it, diff it against what
                     actually ran, show it to a human for approval before any
                     side effect. And it goes stale the instant reality disagrees.

  The fix is a REPLAN step: execute, detect failure, feed the failure back to the
  planner, produce a new plan. That is a conditional edge in module 19's graph -
  and it is where the two approaches converge.

Use Plan-and-Execute when the steps are genuinely knowable and the plan must be
auditable - regulated decisions, anything with a side effect, anything expensive.
Use ReAct when step 2 honestly depends on what step 1 returns.

Module 27 takes the plan idea further: enumerate every call up front and run the
independent ones in PARALLEL.""")

---

**Next:** `27_rewoo_parallel_planning.ipynb`